**Feature Selection Method:** PSO   
**Dataset:** QUT-DV25 (Combined)   
**Developed By:** eDySec Research Team   
**Plartform:** Ubuntu

All experiments in this notebook were conducted using **Python 3.10** with the following libraries:

`pandas==1.5.3`,  
`scikit-learn==1.2.2`,  
`openpyxl`,  
`numpy==1.23.5`,  
`scipy==1.9.3`,  
`tensorflow==2.11.0`,  
`matplotlib==3.7.1`,  
`seaborn==0.12.2`,  
`joblib==1.3.2`,  
`shap==0.41.0`,  
`lime`,  
`flaml[automl]==2.5.0`,  
`notebook==6.5.6`,  
`pywinpty==2.0.10`  (Only for windows)  `threadpoolctl==3.1.0` (for Ubuntu)   
`terminado==0.17.1`,  
`transformers==4.49.0`.

#### Full Environment Setup: https://github.com/tanzirmehedi/eDySec

These versions were used to ensure **consistent and reproducible experimental results**.

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder
from scipy.sparse import hstack
from scipy.sparse import csr_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, confusion_matrix, roc_auc_score, roc_curve,
    precision_recall_fscore_support, cohen_kappa_score
)
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
import matplotlib.pyplot as plt
import seaborn as sns
import time
import joblib
import gc
from sklearn.feature_selection import f_classif
from flaml import AutoML

In [2]:
# This will allow TensorFlow to allocate as much GPU memory as needed, including full 16GB if needed.
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        # Disable memory growth (optional - default is False)
        tf.config.experimental.set_memory_growth(gpus[0], False)
        print("Memory growth disabled (default behavior).")
    except RuntimeError as e:
        print("Error:", e)

Memory growth disabled (default behavior).


In [ ]:
DATASET_NAME = "Combined"
file_path = 'QUT-DV25_'+DATASET_NAME+'_Traces.csv'
data = pd.read_csv(file_path)

In [4]:
gc.collect()
tf.keras.backend.clear_session()


In [5]:
data.head()

,Package_Name,IO_Operations,File_Operations,Network_Operations,Time_Operations,Security_Operations,Process_Operations,Read_Processes,Write_Processes,Read_Data_Transfer,...,Pattern_2,Pattern_3,Pattern_4,Pattern_5,Pattern_6,Pattern_7,Pattern_8,Pattern_9,Pattern_10,Level
0,10Cent10-999.0.4.tar.gz,ioctl,NaN,"getrandom, uname",NaN,NaN,getpid,"opensnoop-bpfcc, filetop-bpfcc, tcpstates-bpfc...","opensnoop-bpfcc, strace, StreamTrans, Classif~...","opensnoop-bpfcc, filetop-bpfcc, pip, gnome-sys...",...,fstat -> ioctl -> lseek,openat -> fstat -> ioctl,newfstatat -> newfstatat -> openat,newfstatat -> openat -> fstat,newfstatat -> newfstatat -> newfstatat -> no-e...,fstat -> ioctl -> lseek -> no-error -> no-fd,openat -> fstat -> ioctl -> no-error -> no-fd,ioctl -> lseek -> lseek -> error=ENOTTY -> no-fd,read -> read -> close -> no-error -> no-fd,1
1,10Cent11-999.0.4.tar.gz,"ioctl, poll",NaN,uname,NaN,NaN,"wait4, getpid","opensnoop-bpfcc, filetop-bpfcc, tcpstates-bpfc...","opensnoop-bpfcc, strace, StreamTrans, Classif~...","opensnoop-bpfcc, filetop-bpfcc, pip, gnome-sys...",...,read -> read -> read,newfstatat -> openat -> fstat,ioctl -> ioctl -> ioctl,newfstatat -> newfstatat -> openat,newfstatat -> newfstatat -> newfstatat -> no-e...,read -> read -> read -> no-error -> no-fd,newfstatat -> openat -> fstat -> no-error -> n...,ioctl -> ioctl -> ioctl -> no-error -> no-fd,fcntl -> fstat -> fcntl -> no-error -> no-fd,1
2,11Cent-999.0.0.tar.gz,"ioctl, poll",pipe2,"getrandom, uname",NaN,NaN,"vfork, wait4, getpid","opensnoop-bpfcc, filetop-bpfcc, tcpstates-bpfc...","opensnoop-bpfcc, strace, StreamTrans, Classif~...","opensnoop-bpfcc, filetop-bpfcc, pip, gnome-sys...",...,newfstatat -> newfstatat -> openat,read -> read -> read,newfstatat -> openat -> fstat,fstat -> ioctl -> lseek,newfstatat -> newfstatat -> newfstatat -> no-e...,newfstatat -> newfstatat -> openat -> no-error...,read -> read -> read -> no-error -> no-fd,openat -> fstat -> ioctl -> no-error -> no-fd,newfstatat -> openat -> fstat -> no-error -> n...,1
3,11Cent-999.0.1.tar.gz,"ioctl, poll",NaN,uname,NaN,NaN,getpid,"opensnoop-bpfcc, filetop-bpfcc, tcpstates-bpfc...","opensnoop-bpfcc, strace, StreamTrans, Classif~...","opensnoop-bpfcc, filetop-bpfcc, pip, gnome-sys...",...,read -> read -> read,newfstatat -> openat -> fstat,openat -> fstat -> fcntl,fstat -> fcntl -> fstat,newfstatat -> newfstatat -> newfstatat -> no-e...,read -> read -> read -> no-error -> no-fd,newfstatat -> openat -> fstat -> no-error -> n...,openat -> fstat -> fcntl -> no-error -> no-fd,fstat -> fcntl -> fstat -> no-error -> no-fd,1
4,11Cent-999.0.2.tar.gz,"poll, ioctl",NaN,uname,NaN,NaN,getpid,"opensnoop-bpfcc, filetop-bpfcc, tcpstates-bpfc...","opensnoop-bpfcc, strace, StreamTrans, Classif~...","opensnoop-bpfcc, filetop-bpfcc, pip, gnome-sys...",...,read -> read -> read,newfstatat -> openat -> fstat,read -> poll -> read,poll -> read -> read,newfstatat -> newfstatat -> newfstatat -> no-e...,read -> read -> read -> no-error -> no-fd,newfstatat -> openat -> fstat -> no-error -> n...,read -> poll -> read -> error=EAGAIN -> no-fd,newfstatat -> newfstatat -> openat -> no-error...,1


In [7]:
X_df = data.drop(columns=['Level']).copy()
y = data['Level']

# Encode label if needed
if y.dtype == 'object' or y.dtype.name == 'category':
    y = LabelEncoder().fit_transform(y)

# Identify categorical columns (strings)
cat_cols = X_df.select_dtypes(include='object').columns
num_cols = X_df.select_dtypes(include=['int64','float64']).columns

# Simple Label Encoding for categorical columns
for col in cat_cols:
    X_df[col] = LabelEncoder().fit_transform(X_df[col].astype(str))

# Now all columns are numeric
X = X_df.values
feature_names = list(X_df.columns)

# =====================
# Fitness Function
# =====================
def fitness_function(selected_features, X_train, y_train, X_val, y_val):
    if np.sum(selected_features) == 0:
        return 0
    selected_cols = np.where(selected_features == 1)[0]
    model = LogisticRegression(max_iter=500)
    model.fit(X_train[:, selected_cols], y_train)
    preds = model.predict(X_val[:, selected_cols])
    return accuracy_score(y_val, preds)

# =====================
# PSO Feature Selection (20 iterations)
# =====================
def pso_feature_selection(X, y, num_particles=5, max_iter=20):
    n_features = X.shape[1]
    particles = np.random.randint(2, size=(num_particles, n_features))
    velocities = np.random.uniform(-1, 1, (num_particles, n_features))

    pbest_positions = particles.copy()
    pbest_scores = np.zeros(num_particles)
    gbest_position = None
    gbest_score = -1

    X_train, X_val, y_train, y_val = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    # Initialize pbest and gbest
    for i in range(num_particles):
        score = fitness_function(particles[i], X_train, y_train, X_val, y_val)
        pbest_scores[i] = score
        if score > gbest_score:
            gbest_score = score
            gbest_position = particles[i].copy()

    w, c1, c2 = 0.7, 1.5, 1.5
    for t in range(max_iter):
        for i in range(num_particles):
            r1, r2 = np.random.rand(), np.random.rand()
            velocities[i] = (
                w * velocities[i]
                + c1 * r1 * (pbest_positions[i] - particles[i])
                + c2 * r2 * (gbest_position - particles[i])
            )
            sigmoid = 1 / (1 + np.exp(-velocities[i]))
            particles[i] = (np.random.rand(n_features) < sigmoid).astype(int)

            score = fitness_function(particles[i], X_train, y_train, X_val, y_val)
            if score > pbest_scores[i]:
                pbest_scores[i] = score
                pbest_positions[i] = particles[i].copy()
                if score > gbest_score:
                    gbest_score = score
                    gbest_position = particles[i].copy()

        selected = [feature_names[i] for i in range(n_features) if gbest_position[i]==1]
        dropped = [feature_names[i] for i in range(n_features) if gbest_position[i]==0]
        print(f"Iteration {t+1}/{max_iter}, Best Accuracy: {gbest_score:.4f}")
        print(f"Selected features: {selected}")
        print(f"Dropped features: {dropped}\n")

    # Final features after all iterations
    final_selected = [feature_names[i] for i in range(n_features) if gbest_position[i]==1]
    final_dropped = [feature_names[i] for i in range(n_features) if gbest_position[i]==0]
    print("===== FINAL RESULTS AFTER 20 ITERATIONS =====")
    print(f"Best Accuracy: {gbest_score:.4f}")
    print(f"Final Selected features: {final_selected}")
    print(f"Final Dropped features: {final_dropped}")
    print(len(final_selected))
    print(len(final_dropped))

    return gbest_position, gbest_score

best_features, best_score = pso_feature_selection(X, y, num_particles=5, max_iter=20)


C:\Users\USER\anaconda3\envs\EEE\lib\site-packages\sklearn\linear_model\_logistic.py:818: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  extra_warning_msg=_LOGISTIC_SOLVER_CONVERGENCE_MSG,
C:\Users\USER\anaconda3\envs\EEE\lib\site-packages\sklearn\linear_model\_logistic.py:818: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.h

Iteration 1/20, Best Accuracy: 0.9576
Selected features: ['IO_Operations', 'File_Operations', 'Network_Operations', 'Process_Operations', 'Read_Processes', 'Write_Processes', 'Write_Data_Transfer', 'File_Access_Processes', 'Total_Dependencies_List', 'Root_DIR_Access', 'Home_DIR_Access', 'User_DIR_Access', 'Etc_DIR_Access', 'Local_IPs_Access', 'Remote_IPs_Access', 'Local_Port_Access', 'Remote_Port_Access', 'Pattern_2', 'Pattern_4', 'Pattern_5', 'Pattern_6', 'Pattern_7', 'Pattern_8', 'Pattern_9']
Dropped features: ['Package_Name', 'Time_Operations', 'Security_Operations', 'Read_Data_Transfer', 'Direct_Dependencies_List', 'Indirect_Dependencies_List', 'Temp_DIR_Access', 'Sys_DIR_Access', 'Other_DIR_Access', 'State_Transition', 'Total_Dependencies', 'Direct_Dependencies', 'Indirect_Dependencies', 'Pattern_1', 'Pattern_3', 'Pattern_10']



C:\Users\USER\anaconda3\envs\EEE\lib\site-packages\sklearn\linear_model\_logistic.py:818: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  extra_warning_msg=_LOGISTIC_SOLVER_CONVERGENCE_MSG,
C:\Users\USER\anaconda3\envs\EEE\lib\site-packages\sklearn\linear_model\_logistic.py:818: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.h

Iteration 2/20, Best Accuracy: 0.9576
Selected features: ['IO_Operations', 'File_Operations', 'Network_Operations', 'Process_Operations', 'Read_Processes', 'Write_Processes', 'Write_Data_Transfer', 'File_Access_Processes', 'Total_Dependencies_List', 'Root_DIR_Access', 'Home_DIR_Access', 'User_DIR_Access', 'Etc_DIR_Access', 'Local_IPs_Access', 'Remote_IPs_Access', 'Local_Port_Access', 'Remote_Port_Access', 'Pattern_2', 'Pattern_4', 'Pattern_5', 'Pattern_6', 'Pattern_7', 'Pattern_8', 'Pattern_9']
Dropped features: ['Package_Name', 'Time_Operations', 'Security_Operations', 'Read_Data_Transfer', 'Direct_Dependencies_List', 'Indirect_Dependencies_List', 'Temp_DIR_Access', 'Sys_DIR_Access', 'Other_DIR_Access', 'State_Transition', 'Total_Dependencies', 'Direct_Dependencies', 'Indirect_Dependencies', 'Pattern_1', 'Pattern_3', 'Pattern_10']



C:\Users\USER\anaconda3\envs\EEE\lib\site-packages\sklearn\linear_model\_logistic.py:818: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  extra_warning_msg=_LOGISTIC_SOLVER_CONVERGENCE_MSG,
C:\Users\USER\anaconda3\envs\EEE\lib\site-packages\sklearn\linear_model\_logistic.py:818: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.h

...

C:\Users\USER\anaconda3\envs\EEE\lib\site-packages\sklearn\linear_model\_logistic.py:818: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  extra_warning_msg=_LOGISTIC_SOLVER_CONVERGENCE_MSG,
C:\Users\USER\anaconda3\envs\EEE\lib\site-packages\sklearn\linear_model\_logistic.py:818: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.h

Iteration 19/20, Best Accuracy: 0.9737
Selected features: ['IO_Operations', 'File_Operations', 'Network_Operations', 'Process_Operations', 'Read_Processes', 'Write_Processes', 'Read_Data_Transfer', 'Write_Data_Transfer', 'File_Access_Processes', 'Total_Dependencies_List', 'Root_DIR_Access', 'Temp_DIR_Access', 'Home_DIR_Access', 'Local_Port_Access', 'Total_Dependencies', 'Pattern_2', 'Pattern_7', 'Pattern_8', 'Pattern_9']
Dropped features: ['Package_Name', 'Time_Operations', 'Security_Operations', 'Direct_Dependencies_List', 'Indirect_Dependencies_List', 'User_DIR_Access', 'Sys_DIR_Access', 'Etc_DIR_Access', 'Other_DIR_Access', 'State_Transition', 'Local_IPs_Access', 'Remote_IPs_Access', 'Remote_Port_Access', 'Direct_Dependencies', 'Indirect_Dependencies', 'Pattern_1', 'Pattern_3', 'Pattern_4', 'Pattern_5', 'Pattern_6', 'Pattern_10']



C:\Users\USER\anaconda3\envs\EEE\lib\site-packages\sklearn\linear_model\_logistic.py:818: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  extra_warning_msg=_LOGISTIC_SOLVER_CONVERGENCE_MSG,
C:\Users\USER\anaconda3\envs\EEE\lib\site-packages\sklearn\linear_model\_logistic.py:818: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.h

Iteration 20/20, Best Accuracy: 0.9737
Selected features: ['IO_Operations', 'File_Operations', 'Network_Operations', 'Process_Operations', 'Read_Processes', 'Write_Processes', 'Read_Data_Transfer', 'Write_Data_Transfer', 'File_Access_Processes', 'Total_Dependencies_List', 'Root_DIR_Access', 'Temp_DIR_Access', 'Home_DIR_Access', 'Local_Port_Access', 'Total_Dependencies', 'Pattern_2', 'Pattern_7', 'Pattern_8', 'Pattern_9']
Dropped features: ['Package_Name', 'Time_Operations', 'Security_Operations', 'Direct_Dependencies_List', 'Indirect_Dependencies_List', 'User_DIR_Access', 'Sys_DIR_Access', 'Etc_DIR_Access', 'Other_DIR_Access', 'State_Transition', 'Local_IPs_Access', 'Remote_IPs_Access', 'Remote_Port_Access', 'Direct_Dependencies', 'Indirect_Dependencies', 'Pattern_1', 'Pattern_3', 'Pattern_4', 'Pattern_5', 'Pattern_6', 'Pattern_10']

===== FINAL RESULTS AFTER 20 ITERATIONS =====
Best Accuracy: 0.9737
Final Selected features: ['IO_Operations', 'File_Operations', 'Network_Operations', 'P

C:\Users\USER\anaconda3\envs\EEE\lib\site-packages\sklearn\linear_model\_logistic.py:818: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  extra_warning_msg=_LOGISTIC_SOLVER_CONVERGENCE_MSG,


In [8]:
selected_features = ['IO_Operations', 'File_Operations', 'Network_Operations', 'Process_Operations', 'Read_Processes', 'Write_Processes', 'Read_Data_Transfer', 'Write_Data_Transfer', 'File_Access_Processes', 'Total_Dependencies_List', 'Root_DIR_Access', 'Temp_DIR_Access', 'Home_DIR_Access', 'Local_Port_Access', 'Total_Dependencies', 'Pattern_2', 'Pattern_7', 'Pattern_8', 'Pattern_9']
